# 8. Submodules

A **submodule** is a repository inside another repository. The outer
project does not store the inner project's files. It stores only **which
commit** of it to use.

```mermaid
flowchart LR
    subgraph barge["barge (your project)"]
        src["hydrostatics.py"]
        pin["vendor/seawater (one commit id)"]
    end
    pin -.->|points at| lib["seawater repository"]
```

Our example: the sea water properties live in their own repository,
`seawater`, maintained by another group. The barge project uses it.

In [ ]:
!rm -rf ~/git-course/08-submodules
!mkdir -p ~/git-course/08-submodules
%cd ~/git-course/08-submodules

## The library

The hydrography group's repository, with its server `seawater.git`:

In [ ]:
!git init --bare seawater.git
!mkdir seawater
%cd seawater

In [ ]:
%%writefile seawater.py
"""Properties of sea water."""


def density(temperature=15.0):
    """Density of sea water in kg/m^3."""
    return 1028.0 - 0.2 * temperature

In [ ]:
!git init
!git add seawater.py
!git commit -m "Add sea water density"
!git remote add origin ../seawater.git
!git push -u origin main

## Our project

In [ ]:
%cd ..
!git init --bare barge.git
!mkdir barge
%cd barge

In [ ]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft

In [ ]:
!git init
!git add hydrostatics.py
!git commit -m "Add volume"
!git remote add origin ../barge.git
!git push -u origin main

## Adding the submodule

`git submodule add <address> <folder>` puts the library in a folder:

In [ ]:
!git -c protocol.file.allow=always submodule add ../seawater.git vendor/seawater

About `-c protocol.file.allow=always`: our "server" is a local folder, and
for safety git does not allow submodules from local folders unless you say
so. With a real address (`https://github.com/...`) you don't need it.

In [ ]:
!git status

Two new things: the folder, and a file `.gitmodules` that remembers where
the library comes from:

In [ ]:
!cat .gitmodules

In [ ]:
!git commit -m "Add the seawater library as a submodule"
!git push

## What was actually saved

`git ls-tree -r` lists everything a commit contains:

In [ ]:
!git ls-tree -r HEAD

`hydrostatics.py` is a `blob` (a file). `vendor/seawater` is a `commit`:
the project saved **one commit id**, not the library's files. That id pins
the exact version of the library we use.

In [ ]:
!git submodule status

## Surprise: a fresh clone is empty

A colleague clones our project:

In [ ]:
%cd ..
!git clone barge.git colleague

In [ ]:
!ls colleague/vendor/seawater

The folder is **empty**. Cloning the project does not download its
submodules. You have to ask:

In [ ]:
%cd colleague
!git -c protocol.file.allow=always submodule update --init

In [ ]:
!ls vendor/seawater

Tip: `git clone --recurse-submodules <address>` does both steps at once.

## The library gets an update

The hydrography group improves their formula and pushes:

In [ ]:
%cd ../seawater

In [ ]:
%%writefile seawater.py
"""Properties of sea water."""


def density(temperature=15.0, salinity=35.0):
    """Density of sea water in kg/m^3."""
    return 1000.0 + 0.8 * salinity - 0.2 * temperature

In [ ]:
!git add seawater.py
!git commit -m "Take salinity into account"
!git push

Our project is not affected: it still uses the old commit. That is the
point of pinning — nothing changes under you without your say.

In [ ]:
%cd ../barge
!git submodule status

To take the new version, update the submodule:

In [ ]:
!git -c protocol.file.allow=always submodule update --remote

In [ ]:
!git diff

The only change in our project: the commit id. Save it like any change:

In [ ]:
!git add vendor/seawater
!git commit -m "Update the seawater library"
!git push

## Summary

| I want to… | Command |
| - | - |
| add a submodule | `git submodule add <address> <folder>` |
| clone a project with its submodules | `git clone --recurse-submodules <address>` |
| fill in submodules after a plain clone | `git submodule update --init` |
| see which commits are pinned | `git submodule status` |
| move to the library's newest commit | `git submodule update --remote` |

Submodules add steps for everyone. Use them when you need to know exactly
which version of a library produced your results. For published Python
packages (`numpy`, `scipy`), use `pip` and a `requirements.txt` instead.

## Exercises

1. In `colleague`, run `git pull` and then
   `git -c protocol.file.allow=always submodule update`. Does it get the
   new library version?
2. Run `cat vendor/seawater/seawater.py` in `barge` before and after the
   update.

That's the course! The [cheat sheet](../docs/09-cheat-sheet.md) has every
command on one page.